In [8]:
import requests
import pandas as pd
import time
from pathlib import Path
import urllib.parse

SERVICE_KEY = urllib.parse.unquote(
    "Vdj8H5rcTtDDiL3rS3wtwvD9li7jiqrfvSwrDhnA3GkXWhSpukUyYIes7ZXfBjNVcJ69YDRtV2cDoY0pZvR5SQ%3D%3D"
)
BASE_URL = "https://apis.data.go.kr/1613000/RTMSDataSvcAptTradeDev/getRTMSDataSvcAptTradeDev"

LAWD_CODES = {
    "gangnam": "11680",
    "seocho": "11650",
}

START_YEAR, END_YEAR = 2016, 2025

In [9]:
def generate_deal_ymd_list(start_year, end_year):
    ymd_list = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            ymd_list.append(f"{year}{month:02d}")
    return ymd_list

deal_ymd_list = generate_deal_ymd_list(START_YEAR, END_YEAR)
print(len(deal_ymd_list), "개월 조회 예정")

120 개월 조회 예정


In [17]:
import xml.etree.ElementTree as ET

def fetch_apt_trade(lawd_cd, deal_ymd, num_of_rows=1000, page_no=1):
    params = {
        "serviceKey": SERVICE_KEY,
        "LAWD_CD": lawd_cd,
        "DEAL_YMD": deal_ymd,
        "pageNo": page_no,
        "numOfRows": num_of_rows,
    }
    response = requests.get(BASE_URL, params=params, timeout=10)
    response.raise_for_status()
    return response.text

In [19]:
def parse_response(xml_text, gu_name, deal_ymd):
    root = ET.fromstring(xml_text)
    result_code = root.findtext("./header/resultCode")
    if result_code != "000":
        print(f"에러: {gu_name} {deal_ymd} - {root.findtext('./header/resultMsg')}")
        return []

    items = root.findall("./body/items/item")
    records = []
    for item in items:
        record = {child.tag: (child.text or "").strip() for child in item}
        record["gu_name"] = gu_name
        record["deal_ymd"] = deal_ymd
        records.append(record)
    return records

In [20]:
all_records = []

for gu_name, lawd_cd in LAWD_CODES.items():
    for deal_ymd in deal_ymd_list:
        xml_text = fetch_apt_trade(lawd_cd, deal_ymd)
        records = parse_response(xml_text, gu_name, deal_ymd)
        all_records.extend(records)
        time.sleep(0.2)
    print(f"{gu_name} 수집 — 누적 {len(all_records)}건")

df = pd.DataFrame(all_records)
print(df.shape)
df.head()

gangnam 수집 — 누적 39167건
seocho 수집 — 누적 69686건
(69686, 34)


,aptDong,aptNm,aptSeq,bonbun,bubun,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,...,roadNmCd,roadNmSeq,roadNmSggCd,roadNmbCd,sggCd,slerGbn,umdCd,umdNm,gu_name,deal_ymd
0,,동양파라곤,11680-3667,0069,0018,2006,,,,"225,000",...,4166828,01,11680,0,11680,,10400,청담동,gangnam,201601
1,,우성7,11680-306,0615,0000,1987,,,,"90,250",...,4166041,01,11680,0,11680,,11400,일원동,gangnam,201601
2,,수서,11680-305,0711,0000,1992,,,,"50,000",...,4166557,01,11680,0,11680,,11400,일원동,gangnam,201601
3,,삼성동중앙하이츠빌리지,11680-507,0014,0001,2004,,,,"89,000",...,4166816,01,11680,0,11680,,10500,삼성동,gangnam,201601
4,,강남한양수자인(4단지),11680-4298,0686,0000,2014,,,,"61,500",...,3122014,02,11680,0,11680,,11200,자곡동,gangnam,201601


In [23]:
for gu_name in LAWD_CODES.keys():
    gu_df = df[df["gu_name"] == gu_name]
    PROJECT_ROOT = Path.cwd().parent
    save_path = PROJECT_ROOT / "data" / "raw" / gu_name
    save_path.mkdir(parents=True, exist_ok=True)
    gu_df.to_csv(save_path / f"{gu_name}_2016_2025.csv", index=False, encoding="utf-8-sig")
    print(f"{gu_name}: {len(gu_df)}건 저장 완료")

gangnam: 39167건 저장 완료
seocho: 30519건 저장 완료
